In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, when, to_json, struct
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.ml import PipelineModel

# 1. Initialize Spark
spark = SparkSession.builder \
    .appName("FlightDelayStreamTransformer") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# 2. Schema
raw_schema = StructType([
    StructField("fl_date", StringType(), True),
    StructField("day_of_week", IntegerType(), True),
    StructField("op_unique_carrier", StringType(), True),
    StructField("origin", StringType(), True),
    StructField("dest", StringType(), True),
    StructField("crs_dep_time", IntegerType(), True),
    StructField("dep_delay", DoubleType(), True),
    StructField("crs_arr_time", IntegerType(), True),
    StructField("arr_delay", DoubleType(), True),
    StructField("crs_elapsed_time", DoubleType(), True),
    StructField("distance", DoubleType(), True),
    StructField("temperature_2m", DoubleType(), True),
    StructField("precipitation", DoubleType(), True),
    StructField("snowfall", DoubleType(), True),
    StructField("weather_code", DoubleType(), True),
    StructField("wind_speed_10m", DoubleType(), True),
    StructField("wind_gusts_10m", DoubleType(), True),
    StructField("surface_pressure", DoubleType(), True),
    StructField("pressure_msl", DoubleType(), True)
])

# 3. Read RAW stream safely
print("📡 Listening to RAW Kafka topic 'flight-stream-raw'...")
raw_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "flight-stream-raw") \
    .option("startingOffsets", "earliest") \
    .option("failOnDataLoss", "false") \
    .load()

parsed_stream = raw_stream.select(from_json(col("value").cast("string"), raw_schema).alias("data")).select("data.*")

# 4. Apply Feature Engineering
print("⚙️ Applying real-time transformations...")
transformed_stream = parsed_stream.withColumn(
    "is_weekend",
    when(col("day_of_week").isin(1, 7), 1).otherwise(0)
).filter(col("crs_elapsed_time") >= 30)

indexer_model = PipelineModel.load("hdfs://namenode:9000/flight_project/gold/string_indexer_pipeline")
clean_indexed_stream = indexer_model.transform(transformed_stream)

clean_indexed_stream = clean_indexed_stream.drop("fl_date", "day_of_week", "op_unique_carrier", "origin", "dest")

# 5. Push clean features to flight-stream-clean
print("📤 Pushing transformed features to 'flight-stream-clean'...")
kafka_output = clean_indexed_stream.select(to_json(struct([clean_indexed_stream[x] for x in clean_indexed_stream.columns])).alias("value"))

query = kafka_output.writeStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("topic", "flight-stream-clean") \
    .option("checkpointLocation", "hdfs://namenode:9000/flight_project/checkpoints/transformer") \
    .start()

query.awaitTermination()

📡 Listening to RAW Kafka topic 'flight-stream-raw'...
⚙️ Applying real-time transformations...
📤 Pushing transformed features to 'flight-stream-clean'...


In [ ]:
# Run this to stop the stream gracefully
for q in spark.streams.active:
    q.stop()

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 45484)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/conda/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/usr/local/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/usr/local/spark/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/usr/local/spark/python/pyspark/accumulators.py", line 271, in accum_updates
    num_updates =